# Bank Marketing — EDA y Modelado (dataset estándar, 17 variables)

Hackathon 6 — predecir si un cliente aceptará un depósito a plazo (`y ∈ {yes, no}`).

Dataset: `bank-full.csv` (UCI Bank Marketing, **17 variables**: 16 features + `y`).
Este es el dataset que el profesor pidió usar como estándar para toda la clase.

También se entrenó, como comparación opcional (+4 puntos de participación),
un segundo modelo con `bank-additional-full.csv` (20 variables) — ver
`notebooks/modeling_extended.ipynb`. Ambos quedan desplegados en paralelo en
Cloud Run.

La lógica reutilizable (features, preprocesamiento, entrenamiento) vive en
`src/` y es la que realmente usa la API — este notebook la ejecuta para
mostrar el proceso y los resultados, no la duplica.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..") / "src"))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

df = pd.read_csv("../data/bank-full.csv", sep=";")
df.shape

(45211, 17)

## 1. Estructura del dataset (17 variables)

In [2]:
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Variable objetivo y desbalance de clases

In [4]:
print(df["y"].value_counts())
print()
print(df["y"].value_counts(normalize=True))

y
no     39922
yes     5289
Name: count, dtype: int64

y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


El dataset está desbalanceado: **~88.3% "no" vs ~11.7% "yes"** (muy similar
a la proporción del dataset extendido). Se usa `class_weight="balanced"` en
los modelos y se prioriza **F1** y **Balanced Accuracy**, además de
**calibrar el umbral de decisión** (ver sección 6) en vez de usar el 0.5 por
defecto.

## 3. Calidad de datos: nulos, duplicados y 'unknown'

In [5]:
print("Missing values (NaN):", df.isna().sum().sum())
print("Duplicated rows:", df.duplicated().sum())
print()
print("Valores 'unknown' por columna categórica:")
for c in df.select_dtypes(include="object").columns:
    n = (df[c] == "unknown").sum()
    if n > 0:
        print(f"  {c}: {n} ({n/len(df)*100:.1f}%)")

Missing values (NaN): 0
Duplicated rows: 0

Valores 'unknown' por columna categórica:
  job: 288 (0.6%)
  education: 1857 (4.1%)
  contact: 13020 (28.8%)
  poutcome: 36959 (81.7%)


C:\Users\bihondaepiquien\AppData\Local\Temp\ipykernel_39752\922973187.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include="object").columns:


No hay `NaN` ni filas duplicadas exactas en esta versión del dataset. Al
igual que en la versión extendida, varias columnas categóricas codifican
"no informado" como la categoría `"unknown"` en vez de un nulo explícito
(hasta 29% en `poutcome`); se mantienen como una categoría más vía One-Hot
Encoding.

## 4. Variables numéricas y categóricas

In [6]:
from preprocessing import STANDARD_NUMERIC_FEATURES, STANDARD_CATEGORICAL_FEATURES, STANDARD_FEATURES, LEAKAGE_FEATURES

print("Numéricas:", STANDARD_NUMERIC_FEATURES)
print()
print("Categóricas:", STANDARD_CATEGORICAL_FEATURES)
print()
print("Excluidas por leakage:", LEAKAGE_FEATURES)
print()
print("Total features finales:", len(STANDARD_FEATURES))

Numéricas: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Excluidas por leakage: ['duration']

Total features finales: 15


In [7]:
df[STANDARD_NUMERIC_FEATURES].describe()

,age,balance,day,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,63.000000,871.000000,275.000000


### Nota sobre `pdays`

En esta versión del dataset, `pdays` usa el centinela **`-1`** (no `999`
como en `bank-additional-full.csv`) para "el cliente nunca fue contactado
antes de esta campaña". No es un dato faltante real, es una codificación
propia del dataset (ver `data/bank-names.txt`).

In [8]:
print((df["pdays"] == -1).mean())
df.groupby("y")["pdays"].apply(lambda s: (s == -1).mean())

0.8173674548229414


y
no     0.840890
yes    0.639818
Name: pdays, dtype: float64

## 5. Análisis de leakage — por qué se excluye `duration`

In [9]:
# duration = duración de la llamada en segundos. Se conoce SOLO después de
# que la llamada ya terminó -> en producción, al momento de decidir a quién
# llamar, este dato todavía no existe. Usarlo sería data leakage.
print(df.groupby("y")["duration"].mean())
print()
print(df.groupby("y")["duration"].median())

y
no     221.182806
yes    537.294574
Name: duration, dtype: float64

y
no     164.0
yes    426.0
Name: duration, dtype: float64


Igual que en el dataset extendido, `duration` está fuertemente
correlacionada con el resultado precisamente porque es un efecto posterior
a la decisión del cliente, no una causa disponible de antemano. Se excluye
completamente de `STANDARD_FEATURES` y se verifica de forma explícita y
programática antes de entrenar (`src/train_common.py`,
`assert "duration" not in features`) y en cada request de la API
(`api/main.py` rechaza el campo `duration` con HTTP 422).

## 6. Entrenamiento, calibración de umbral y comparación de modelos

El entrenamiento real (búsqueda pequeña de hiperparámetros, calibración del
umbral de decisión, comparación de métricas y guardado del modelo final) se
ejecuta con:

```
python src/train.py
```

A continuación se reproduce el mismo flujo dentro del notebook.

In [10]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, confusion_matrix
from sklearn.base import clone

from preprocessing import build_pipeline, TARGET

df_clean = df.drop_duplicates()
X = df_clean[STANDARD_FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
X_train.shape, X_val.shape, X_test.shape

((36168, 15), (7234, 15), (9043, 15))

In [11]:
lr_pipeline = build_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), variant="standard"
)
lr_grid = GridSearchCV(
    lr_pipeline, param_grid={"model__C": [0.1, 1.0, 3.0]}, scoring="f1_macro", cv=3, n_jobs=-1
)
lr_grid.fit(X_train, y_train)
lr_grid.best_params_

{'model__C': 3.0}

In [12]:
rf_pipeline = build_pipeline(
    RandomForestClassifier(class_weight="balanced", min_samples_leaf=10, random_state=42, n_jobs=-1),
    variant="standard",
)
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid={"model__n_estimators": [200, 300], "model__max_depth": [8, 10, 12]},
    scoring="f1_macro", cv=3, n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
rf_grid.best_params_

{'model__max_depth': 12, 'model__n_estimators': 300}

### Por qué calibrar el umbral de decisión

Con ~12% de casos positivos, el umbral por defecto (0.5) de `.predict()`
subestima sistemáticamente la clase "yes". Se busca, sobre un split de
**validación** (separado del propio `X_train`, nunca del `X_test`), el
umbral que maximiza F1 usando la curva precision-recall. El modelo usado
para reportar métricas de test se reentrena luego sobre el `X_train`
completo -- el split de validación solo se usa para elegir el umbral, nunca
para seleccionar hiperparámetros ni para el resultado final reportado.

In [13]:
def yes_proba(model, X):
    classes = list(model.classes_)
    return model.predict_proba(X)[:, classes.index("yes")]

def tune_threshold(model, X_val, y_val):
    proba = yes_proba(model, X_val)
    precision, recall, thresholds = precision_recall_curve((y_val == "yes").astype(int), proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-9)
    best_idx = int(np.nanargmax(f1s[:-1]))
    return float(thresholds[best_idx])

def evaluate(model, X_test, y_test, threshold):
    proba = yes_proba(model, X_test)
    y_pred = np.where(proba >= threshold, "yes", "no")
    return {
        "f1_score": f1_score(y_test, y_pred, pos_label="yes"),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score((y_test == "yes").astype(int), proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=["no", "yes"]),
    }

results = {}
for name, grid in [("logistic_regression", lr_grid), ("random_forest", rf_grid)]:
    val_model = clone(grid.best_estimator_)
    val_model.fit(X_tr2, y_tr2)
    threshold = tune_threshold(val_model, X_val, y_val)

    test_model = clone(grid.best_estimator_)
    test_model.fit(X_train, y_train)
    metrics = evaluate(test_model, X_test, y_test, threshold)
    metrics["decision_threshold"] = threshold
    results[name] = metrics
    print(name, {k: v for k, v in metrics.items() if k != "confusion_matrix"})

logistic_regression {'f1_score': 0.4505050505050505, 'balanced_accuracy': 0.680969161222661, 'roc_auc': 0.7722396554030301, 'decision_threshold': 0.663317482039106}


random_forest {'f1_score': 0.4732180514550822, 'balanced_accuracy': 0.7180345828011643, 'roc_auc': 0.7996381447728669, 'decision_threshold': 0.5901678670644097}


## 7. Modelos finales (ambos se despliegan)

Se compara F1 y Balanced Accuracy (con umbral calibrado) en el conjunto de
test (hold-out). El requisito de la Hackathon es entrenar y comparar al
menos 2 modelos -- en vez de descartar el que no gana, **ambos candidatos
se reentrenan con el 100% de los datos disponibles** y se sirven en la API
(`POST /predict` para Random Forest, `POST /predict/standard/logistic_regression`
para Logistic Regression), cada uno con su propio umbral calibrado.

```
model/standard/logistic_regression/model.joblib  + model_info.json
model/standard/random_forest/model.joblib        + model_info.json
model/standard/comparison.json                    (metricas de ambos + cual se recomendaria)
```

In [14]:
import json

with open("../model/standard/comparison.json") as f:
    comparison = json.load(f)

print("Dataset:", comparison["dataset"])
print("Modelo recomendado (si solo se desplegara uno):", comparison["recommended_model"])
print()
for model_key, metrics in comparison["all_candidates"].items():
    print(f"--- {model_key} ---")
    for k, v in metrics.items():
        if k not in ("confusion_matrix", "confusion_matrix_labels"):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Dataset: standard (bank-full.csv, 17 variables)
Modelo recomendado (si solo se desplegara uno): random_forest

--- logistic_regression ---
  decision_threshold: 0.6633
  f1_score: 0.4505
  balanced_accuracy: 0.6810
  precision: 0.4837
  recall: 0.4216
  roc_auc: 0.7722
--- random_forest ---
  decision_threshold: 0.5902
  f1_score: 0.4732
  balanced_accuracy: 0.7180
  precision: 0.4273
  recall: 0.5302
  roc_auc: 0.7996


## 8. Comparación con el dataset extendido (bonus)

El dataset extendido (`bank-additional-full.csv`, 20 variables, ver
`modeling_extended.ipynb`) agrega indicadores macroeconómicos
(`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`,
`nr.employed`) que este dataset estándar no tiene. Esa señal adicional
explica la diferencia de desempeño entre ambos modelos finales (ver README,
sección 5, tabla comparativa).

## 9. Verificación explícita: `duration` fuera del modelo final

In [15]:
assert "duration" not in STANDARD_FEATURES
print("OK: duration NOT IN final_features")

OK: duration NOT IN final_features
